In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from transformers import AutoModelForCausalLM
from geomechinterp.causal.mygpt import SymbolTokenizer
from geomechinterp.causal.base_functions import ALL_SYMBOLS, EXTRA_SYMBOLS
import transformer_lens as tl
import torch


model = AutoModelForCausalLM.from_pretrained("../gpt_rope_custom/checkpoint-407000/")
tokenizer = SymbolTokenizer(ALL_SYMBOLS + EXTRA_SYMBOLS)

In [4]:
from datasets import load_from_disk
test_dataset = load_from_disk("../tokenized_test")

In [5]:
from geomechinterp.causal.mygpt import DataCollator

data_collator = DataCollator(test_dataset)  

In [6]:
from geomechinterp.causal.base_functions import position_parity_check, ab_f, case_f, f12_f, plus_minus_f, qm_f, bracket_f

all_binary_generators = {
    "position_parity_check": position_parity_check,  # no control!
    "ab_f": ab_f,
    "case_f": case_f,
    "f12_f": f12_f,
    "plus_minus_f": plus_minus_f,
    "qm_f": qm_f,
    "bracket_f": bracket_f,
}

In [7]:
from functools import partial
import torch
from transformer_lens import ActivationCache, HookedTransformer, HookedTransformerConfig
from geomechinterp.causal.mygpt import get_model_config

In [8]:
from transformer_lens.loading_from_pretrained import convert_hf_model_config
from geomechinterp.tflens.utils import convert_nanogpt_weights, load_gpt2_to_hooked_transformer

# converted_config = get_model_config(vocab_size=len(ALL_SYMBOLS)+len(EXTRA_SYMBOLS))

hook_config = HookedTransformerConfig(d_vocab=29, n_ctx=128, d_model=128,
                                      d_head=32, n_layers=6, n_heads=4,
                                      rotary_dim=64, act_fn='gelu_new', 
                                      original_architecture='GPT2LMHeadModel')

In [9]:
hooked_model = HookedTransformer(hook_config)

In [10]:
model.transformer.h[0].attn.c_attn.weight.shape

torch.Size([128, 384])

In [11]:
# Define HookedTransformer config
hook_config = HookedTransformerConfig(d_vocab=29, n_ctx=128, d_model=128,
                                      d_head=32, n_layers=6, n_heads=4,
                                      act_fn='gelu_new', 
                                      # rotary_dim=64,
                                      positional_embedding_type='standard')

 
hooked_model_1 = load_gpt2_to_hooked_transformer(model, hook_config)

hooked_model_dict = convert_nanogpt_weights(model.state_dict(), hook_config, bias=True)
hooked_model_2 = HookedTransformer(hook_config)
hooked_model_2.load_state_dict(hooked_model_dict)
# ensure that both models return the same output for the same input
model.to("mps")
hooked_model_1.to("mps")
hooked_model_2.to("mps")

input_ids = torch.tensor([1, 2, 3, 3, 1, 2, 5, 6, 2, 10, 2, 3, 15, 18, 20, 7]).to("mps")


model.eval()
hooked_model_1.eval()
hooked_model_2.eval()

out1 = model(input_ids)
out2, hooked_act = hooked_model_1.run_with_cache(input_ids)
out3, hooked_act = hooked_model_2.run_with_cache(input_ids)

print(torch.allclose(out1['logits'], out2.squeeze()))
print(torch.allclose(out1['logits'], out3.squeeze()))
print(torch.allclose(out2, out3.squeeze()))

k = 1

print('---'*10)
for k in range(out2.shape[1]):
    # use cosine similarity after softmax
    print(torch.cosine_similarity(torch.nn.functional.softmax(out1['logits'][k,:], dim=-1), torch.nn.functional.softmax(out2[:,k,:], dim=-1)))    



# print(out2[:,k,:])
# print(out3[:,k,:])
# for k in range(out2.shape[1]):
#     # use cosine similarity after softmax
#     print(torch.cosine_similarity(torch.nn.functional.softmax(out2[:,k,:], dim=-1), torch.nn.functional.softmax(out3[:,k,:], dim=-1)))    

# print('---'*10)
# for k in range(out2.shape[1]):
#     # use cosine similarity after softmax
#     print(torch.cosine_similarity(torch.nn.functional.softmax(out1['logits'][k,:], dim=-1), torch.nn.functional.softmax(out3[:,k,:], dim=-1)))    


Moving model to device:  mps
Moving model to device:  mps
False
False
False
------------------------------
tensor([0.9994], device='mps:0', grad_fn=<SumBackward1>)
tensor([1.], device='mps:0', grad_fn=<SumBackward1>)
tensor([1.], device='mps:0', grad_fn=<SumBackward1>)
tensor([1.0000], device='mps:0', grad_fn=<SumBackward1>)
tensor([1.], device='mps:0', grad_fn=<SumBackward1>)
tensor([1.], device='mps:0', grad_fn=<SumBackward1>)
tensor([0.9523], device='mps:0', grad_fn=<SumBackward1>)
tensor([1.], device='mps:0', grad_fn=<SumBackward1>)
tensor([0.9994], device='mps:0', grad_fn=<SumBackward1>)
tensor([0.9347], device='mps:0', grad_fn=<SumBackward1>)
tensor([1.], device='mps:0', grad_fn=<SumBackward1>)
tensor([1.], device='mps:0', grad_fn=<SumBackward1>)
tensor([0.9226], device='mps:0', grad_fn=<SumBackward1>)
tensor([0.9750], device='mps:0', grad_fn=<SumBackward1>)
tensor([0.9235], device='mps:0', grad_fn=<SumBackward1>)
tensor([0.9390], device='mps:0', grad_fn=<SumBackward1>)


In [12]:
def get_module_group_counts(model):
    """Get parameter counts grouped by module type."""
    counts = {
        'embedding': 0,
        'attention': 0, 
        'mlp': 0,
        'layer_norm': 0,
        'output': 0
    }
    
    for name, param in model.named_parameters():
        if 'wte' in name or 'wpe' in name or ('embed' in name and not 'unembed' in name):
            counts['embedding'] += param.numel()
        elif any(x in name for x in ['attn', 'attention']):
            counts['attention'] += param.numel()
        elif any(x in name for x in ['mlp', 'c_fc', 'c_proj']):
            counts['mlp'] += param.numel()
        elif 'ln' in name:
            counts['layer_norm'] += param.numel()
        elif any(x in name for x in ['lm_head', 'unembed', 'ln_final', 'ln_f']):
            counts['output'] += param.numel()
    
    if getattr(model, 'lm_head', None) is not None:
        counts['output'] += model.lm_head.weight.numel()
    return counts

# Get grouped counts for both models
hf_counts = get_module_group_counts(model)
hooked_counts = get_module_group_counts(hooked_model)

# Print comparison table
print("Module Group Parameter Counts")
print("-" * 60)
print(f"{'Module Type':<20} | {'HuggingFace':>12} | {'HookedTransformer':>12} | {'Delta':>8}")
print("-" * 60)

total_hf = 0
total_hooked = 0

for module in ['embedding', 'attention', 'mlp', 'layer_norm', 'output']:
    hf = hf_counts[module]
    hooked = hooked_counts[module]
    delta = hooked - hf
    total_hf += hf
    total_hooked += hooked
    print(f"{module:<20} | {hf:>12,} | {hooked:>12,} | {delta:>8,}")

print("-" * 60)
print(f"{'TOTAL':<20} | {total_hf:>12,} | {total_hooked:>12,} | {total_hooked-total_hf:>8,}")


Module Group Parameter Counts
------------------------------------------------------------
Module Type          |  HuggingFace | HookedTransformer |    Delta
------------------------------------------------------------
embedding            |       20,096 |       20,096 |        0
attention            |      396,288 |      396,288 |        0
mlp                  |      790,272 |      790,272 |        0
layer_norm           |        3,328 |        3,328 |        0
output               |        3,712 |        3,741 |       29
------------------------------------------------------------
TOTAL                |    1,213,696 |    1,213,725 |       29


In [13]:
# Positional Embedding Validation
hf_pos_emb = model.state_dict()["transformer.wpe.weight"]
hook_pos_emb = hooked_model_1.state_dict()["pos_embed.W_pos"]
assert torch.allclose(hf_pos_emb, hook_pos_emb), "Mismatch in positional embeddings"

In [14]:
# Validate MLP weights
for i in range(hook_config.n_layers):
    hf_w_in = model.state_dict()[f"transformer.h.{i}.mlp.c_fc.weight"]
    hook_w_in = hooked_model_1.state_dict()[f"blocks.{i}.mlp.W_in"]
    assert torch.allclose(hf_w_in, hook_w_in), f"Mismatch in W_in for layer {i}"

    hf_b_in = model.state_dict()[f"transformer.h.{i}.mlp.c_fc.bias"]
    hook_b_in = hooked_model_1.state_dict()[f"blocks.{i}.mlp.b_in"]
    assert torch.allclose(hf_b_in, hook_b_in), f"Mismatch in b_in for layer {i}"

    hf_w_out = model.state_dict()[f"transformer.h.{i}.mlp.c_proj.weight"]
    hook_w_out = hooked_model_1.state_dict()[f"blocks.{i}.mlp.W_out"]
    assert torch.allclose(hf_w_out, hook_w_out), f"Mismatch in W_out for layer {i}"

    hf_b_out = model.state_dict()[f"transformer.h.{i}.mlp.c_proj.bias"]
    hook_b_out = hooked_model_1.state_dict()[f"blocks.{i}.mlp.b_out"]
    assert torch.allclose(hf_b_out, hook_b_out), f"Mismatch in b_out for layer {i}"

In [15]:
def split_heads(tensor, num_heads, attn_head_size):
    """
    Splits hidden_size dim into attn_head_size and num_heads
    """
    new_shape = tensor.size()[:-1] + (num_heads, attn_head_size)
    tensor = tensor.view(new_shape)
    return tensor.permute(0, 2, 1, 3)  # (batch, head, seq_length, head_features)

In [16]:
c_attn_1 = model.transformer.h[0].attn.c_attn
# c_attn_2 = hooked_model_1.blocks[0].attn.c_attn

random_input = torch.randn(1, 3, 128).to("mps")
query, key, value = c_attn_1(random_input).split(128, dim=2)
print(query.shape, key.shape, value.shape)

query_2, key_2, value_2 = hooked_model_1.blocks[0].attn.calculate_qkv_matrices(random_input, random_input, random_input)
print(query_2.shape, key_2.shape, value_2.shape)

query = split_heads(query, 4, 32).permute(0,2,1,3)
key = split_heads(key, 4, 32).permute(0,2,1,3)
value = split_heads(value, 4, 32).permute(0,2,1,3)

print(query.shape, key.shape, value.shape)

print(torch.dist(query, query_2))
print(torch.dist(key, key_2))
print(torch.dist(value, value_2))
assert torch.allclose(query, query_2, rtol=1e-3)
assert torch.allclose(key, key_2, rtol=1e-3)
# passes but only with low tolerance!
assert torch.allclose(value, value_2, rtol=1e-4)


torch.Size([1, 3, 128]) torch.Size([1, 3, 128]) torch.Size([1, 3, 128])
torch.Size([1, 3, 4, 32]) torch.Size([1, 3, 4, 32]) torch.Size([1, 3, 4, 32])
torch.Size([1, 3, 4, 32]) torch.Size([1, 3, 4, 32]) torch.Size([1, 3, 4, 32])
tensor(2.2466e-06, device='mps:0', grad_fn=<DistBackward0>)
tensor(2.5221e-06, device='mps:0', grad_fn=<DistBackward0>)
tensor(1.0888e-06, device='mps:0', grad_fn=<DistBackward0>)


In [17]:
print(torch.allclose(model.transformer.h[0].ln_1.weight, hooked_model_1.blocks[0].ln1.w))
print(torch.allclose(model.transformer.h[0].ln_1.bias, hooked_model_1.blocks[0].ln1.b))

True
True


In [18]:
L1 = model.transformer.h[0]
L2 = hooked_model_1.blocks[0]

# print(L1)
# print('----------')
# print(L2)

test_input = torch.randn(1, 3, 128).to("mps")
out1 = L1(test_input)
out2 = L2(test_input)

print(torch.allclose(out1[0].squeeze(), out2.squeeze()))

attn_out1 = L1.attn(test_input)[0].squeeze()
attn_out2 = L2.attn(query_input=test_input, key_input=test_input, value_input=test_input)

print(torch.allclose(attn_out1, attn_out2, rtol=1e-1))

assert torch.allclose(L1.ln_1(test_input), L2.ln1(test_input))
assert torch.allclose(L1.ln_2(test_input), L2.ln2(test_input))
assert torch.allclose(L1.mlp(test_input), L2.mlp(test_input))

torch.allclose(L1.attn.c_attn.bias, torch.cat([L2.attn.b_Q.view(-1), L2.attn.b_K.view(-1), L2.attn.b_V.view(-1)]))

c_proj = L1.attn.c_attn.weight
c_proj_qkv = torch.cat([L2.attn.W_Q.permute(1,0,2).reshape(128,-1),
        L2.attn.W_K.permute(1,0,2).reshape(128,-1),
        L2.attn.W_V.permute(1,0,2).reshape(128,-1)], dim=1)
torch.allclose(c_proj, c_proj_qkv)

False
False


True

In [19]:
# Get QKV from L1 by applying weight and bias
qkv = test_input.squeeze()@L1.attn.c_attn.weight + L1.attn.c_attn.bias
q, k, v = qkv.split(qkv.shape[1]//3, dim=1)

# Get QKV from L2 using its calculate_qkv_matrices method
q2, k2, v2 = L2.attn.calculate_qkv_matrices(test_input, test_input, test_input)

print(q.shape, q2.shape)
print(torch.allclose(q, q2.reshape(q.shape), rtol=1e-4))
print(torch.allclose(k, k2.reshape(k.shape), rtol=1e-4))
print(torch.allclose(v, v2.reshape(v.shape), rtol=1e-4))

torch.Size([3, 128]) torch.Size([1, 3, 4, 32])
True
True
True


In [20]:
from transformer_lens.utilities.attention import simple_attn_linear

assert torch.allclose(q.reshape(q.shape[0], q2.shape[2], q2.shape[3]), simple_attn_linear(test_input, L2.attn.W_Q, L2.attn.b_Q), rtol=1e-5)

In [21]:
assert torch.allclose(model.transformer.h[2].mlp.c_proj.weight, hooked_model_1.blocks[2].mlp.W_out)
model.transformer.h[2].mlp.c_proj.weight.dtype, hooked_model_1.blocks[2].mlp.W_out.dtype

(torch.float32, torch.float32)

In [147]:
# Dictionary to store outputs from hooks
outputs = {}

# Define hooks for each component
def embedding_hook(module, input, output):
    outputs["embedding"] = output

def attn_hook(module, input, output):
    layer_name = f"h.{next(i for i,m in enumerate(model.transformer.h) if m.attn == module)}"
    outputs[f"{layer_name}.attn"] = output

def mlp_hook(module, input, output):
    layer_name = f"h.{next(i for i,m in enumerate(model.transformer.h) if m.mlp == module)}"
    outputs[f"{layer_name}.mlp"] = output

def ln1_hook(module, input, output):
    layer_name = f"h.{next(i for i,m in enumerate(model.transformer.h) if m.ln_1 == module)}"
    outputs[f"{layer_name}.ln1"] = output

def ln2_hook(module, input, output):
    layer_name = f"h.{next(i for i,m in enumerate(model.transformer.h) if m.ln_2 == module)}"
    outputs[f"{layer_name}.ln2"] = output

# Register hooks
hooks = []
hooks.append(model.transformer.wte.register_forward_hook(embedding_hook))

for layer in model.transformer.h:
    hooks.append(layer.ln_1.register_forward_hook(ln1_hook))
    hooks.append(layer.attn.register_forward_hook(attn_hook))
    hooks.append(layer.ln_2.register_forward_hook(ln2_hook))
    hooks.append(layer.mlp.register_forward_hook(mlp_hook))

# Run forward pass
out = model(input_ids)

# Compare activations
print("Comparing activations between models:")

# Compare embeddings
emb_match = torch.allclose(outputs["embedding"], hooked_act["hook_embed"])
print(f"Embedding match: {emb_match}")

# Compare each layer's activations
for i in range(len(model.transformer.h)):
    ln1_match = torch.dist(outputs[f"h.{i}.ln1"], hooked_act[f"blocks.{i}.ln1.hook_normalized"]) # rtol=1e-2)
    attn_match = torch.dist(outputs[f"h.{i}.attn"][0], hooked_act[f"blocks.{i}.hook_attn_out"])
    ln2_match = torch.dist(outputs[f"h.{i}.ln2"], hooked_act[f"blocks.{i}.ln2.hook_normalized"])
    mlp_match = torch.dist(outputs[f"h.{i}.mlp"][0], hooked_act[f"blocks.{i}.hook_mlp_out"])
    
    print(f"\nLayer {i}:")
    print(f"  LN1 match: {ln1_match}")
    print(f"  Attention match: {attn_match}")
    print(f"  LN2 match: {ln2_match}")
    print(f"  MLP match: {mlp_match}")

# Remove hooks
for hook in hooks:
    hook.remove()



Comparing activations between models:
Embedding match: True

Layer 0:
  LN1 match: 1.2197841670058551e-06
  Attention match: 19.362913131713867
  LN2 match: 33.90852355957031
  MLP match: 64.2952880859375

Layer 1:
  LN1 match: 12.866151809692383
  Attention match: 58.39638900756836
  LN2 match: 28.323471069335938
  MLP match: 62.11837387084961

Layer 2:
  LN1 match: 19.40070343017578
  Attention match: 62.66394805908203
  LN2 match: 32.296958923339844
  MLP match: 46.508209228515625

Layer 3:
  LN1 match: 20.690961837768555
  Attention match: 80.60347747802734
  LN2 match: 36.61201095581055
  MLP match: 94.75929260253906

Layer 4:
  LN1 match: 24.010698318481445
  Attention match: 95.31670379638672
  LN2 match: 41.456050872802734
  MLP match: 130.28070068359375

Layer 5:
  LN1 match: 23.366609573364258
  Attention match: 146.72801208496094
  LN2 match: 50.47832489013672
  MLP match: 1071.9420166015625


In [ ]:
'blocks.0.hook_resid_pre',
'blocks.0.ln1.hook_scale',
'blocks.0.ln1.hook_normalized',
'blocks.0.attn.hook_q',
'blocks.0.attn.hook_k',
'blocks.0.attn.hook_v',
'blocks.0.attn.hook_attn_scores',
'blocks.0.attn.hook_pattern',
'blocks.0.attn.hook_z',
'blocks.0.hook_attn_out', 
'blocks.0.hook_resid_mid', 
'blocks.0.ln2.hook_scale',
'blocks.0.ln2.hook_normalized', 
'blocks.0.mlp.hook_pre', 
'blocks.0.mlp.hook_post', 
'blocks.0.hook_mlp_out', 
'blocks.0.hook_resid_post'